# 06 -- MOFA+ Joint Multi-Omic Factor Analysis

MOFA+ jointly decomposes microbiome and metabolome data into shared latent
factors. Each factor represents an axis of co-variation across the two omics
views (species abundances and metabolite concentrations). We fit a
**single-group** MOFA+ model on all samples, prune to genuinely active factors, canonicalize
factor sign, test each active factor against CRC stage (Kruskal-Wallis +
Spearman, BH-FDR corrected), and save the **full** loadings/scores/
association tables for downstream use by `08_pathway_analysis.ipynb` and
`09_pathway_network.ipynb`.

**Reads:** `RESULTS_DIR/intermediate/preprocessed_data_reduced_v2.pkl` (NB02 output).

**Writes** (exact filenames NB08/NB09 depend on):
- `RESULTS_DIR/tables/mofa_factor_stage_association_v2.csv` -- KW + Spearman
  vs `Stage.Group` / `Stage.Group.Numeric`, BH-FDR corrected across all tested factors.
- `RESULTS_DIR/tables/mofa_factor_loadings_v2.csv` -- **every** active factor
  x **every** feature in both views (not a top-3/top-10 filtered subset).
- `RESULTS_DIR/tables/mofa_factor_scores_v2.csv` -- `Sample` + every `FactorN`
  + `Study.Group` + `CRC.Stage.Numeric` + `Stage.Group` (+ `Stage.Group.Numeric` bonus).

## Sign-canonicalization convention (read this if you are NB08/NB09)

MOFA+'s factor sign and scale are statistically arbitrary -- the model is
invariant to flipping a factor's sign and negating its loadings at the same
time. Before anything is saved, **every** factor's sign is fixed so that,
consistently:

> **A higher `FactorN` score means more advanced CRC disease state**
> (correlated positively with `CRC.Stage.Numeric`, the fine-grained corrected
> ordinal encoding from `utils.CRC_STAGE_NUMERIC`: Healthy=0, Stage_0=1,
> Stage_I_II=2, Stage_III_IV=3. MP is excluded from the study and has no entry; HS is carried as `PostSurgery` with a NaN ordinal position, so neither can enter this correlation).

## Why a single group (not one MOFA+ group per stage)

MOFA+ centres every feature **within each group**. With groups = stage categories, each
stage's mean is removed before the factors are learned, so between-stage differences --
exactly what the stage-association test below looks for -- cannot appear in any factor. A
single group keeps them, and the stage test is then meaningful.

Concretely: for each factor, we Spearman-correlate the raw factor score
against `CRC.Stage.Numeric`; if the correlation is negative, we flip the
sign of that factor's scores **and** its corresponding species/metabolite
loadings together, so the score/loading relationship stays self-consistent.
This means a **positive** `Weight` for a feature on `FactorN` in
`mofa_factor_loadings_v2.csv` indicates that feature trends with the
more-advanced-disease direction of that factor, for every factor, uniformly.

## Bugs fixed vs. the old `exploratory/06_multiomics_factors.ipynb`

1. Hardcoded dead path (`c:\Users\andna\Documents\KI\Notebooks`) and a
   wrong-drive input pickle -- replaced with `sys.path.insert(0, '.')` and
   `RESULTS_DIR` from `utils`.
2. `calculate_variance_explained()` indexing -- verified directly against
   the installed `mofapy2` source (`mofapy2/core/BayesNet.py`): it returns a
   **list of length `len(GROUPS)`**, each element an `(n_views, K)` array,
   i.e. indexed `[group][view]`. The old code's `.get('r2_per_factor', ...)`
   dict-style access raised `AttributeError`, silently swallowed by a broad
   `try/except`, so the variance-explained and "factor activity" figures
   never actually rendered on a real MOFA+ fit.
3. Undetected metabolite values are converted to `NaN` (MOFA+'s native
   missing-value handling) using NB02's `undetected_mtb` mask (pre-imputation value
   missing or <= 0). The earlier "tied-for-column-minimum" detection changed 0 cells,
   because per-sample median centring breaks the ties.
4. Factor count is no longer hardcoded: we fit with a generous ceiling
   (15) and explicitly prune to active factors via `removeInactiveFactors()`
   -- looped, because (verified from source) that method removes only ONE
   factor per call by design.
5. BH-FDR correction (`statsmodels.stats.multitest.multipletests`) is
   applied across all tested factors' KW and Spearman p-values before
   calling anything "significant".
6. **Every** active factor's loadings are saved (both views, no top-3/top-10
   filter) -- the hard NB08 contract requirement, since NB08 hardcodes
   specific `FactorN` columns that must actually exist in this file.
7. Factor sign is canonicalized against `CRC.Stage.Numeric` (see above).
8. All stage/severity encoding uses `utils.CRC_STAGE_ORDER` /
   `utils.STAGE_GROUP_MAP` / `utils.STAGE_GROUP_ORDER` -- no local redefinition.
9. Every output directory is created with `.mkdir(parents=True,
   exist_ok=True)` immediately before the `savefig()`/`to_csv()` that uses it.


In [ ]:
import sys
sys.path.insert(0, '.')

import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore', category=FutureWarning)

import utils
import viz
from utils import (
    RESULTS_DIR,
    CRC_STAGE_ORDER, CRC_STAGE_NUMERIC,
    STAGE_GROUP_MAP, STAGE_GROUP_ORDER, STAGE_GROUP_NUMERIC,
)

for sub in ['tables', 'intermediate', 'figures/multiomics']:
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded successfully.')
print(f'RESULTS_DIR = {RESULTS_DIR}')


## 1. Load NB02's reduced species + metabolite matrices


In [ ]:
CRC_DS = 'YACHIDA-CRC-2019'  # only Yachida carries CRC.Stage.Numeric / Stage.Group

reduced_pkl_path = RESULTS_DIR / 'intermediate' / 'preprocessed_data_reduced_v2.pkl'
with open(reduced_pkl_path, 'rb') as f:
    _prep = pickle.load(f)

assert CRC_DS in _prep['datasets'], f'{CRC_DS} not in reduced dataset list: {_prep["datasets"]}'

spc     = _prep['transformed_species'][CRC_DS].copy()
mtb     = _prep['transformed_mtb'][CRC_DS].copy()
meta    = _prep['harmonized_meta'][CRC_DS].copy()
pa_cols = _prep['polyamine_columns'][CRC_DS]

print(f'Loaded from: {reduced_pkl_path}')
print(f'Species matrix    : {spc.shape}  (Sample + {spc.shape[1]-1} species)')
print(f'Metabolite matrix : {mtb.shape}  (Sample + {mtb.shape[1]-1} metabolites)')
print(f'Metadata          : {meta.shape}')
print(f'Polyamine columns found: {list(pa_cols.values())}')


## 2. Undetected metabolite values -> `NaN` (bug fix #3)

NB01's metabolite transform is `impute_half_min()` -> `log10(x+1)` -> per-sample median
centring, so undetected cells end up as a sample-shifted imputed value rather than a
recognisable constant. Instead of trying to recover them from the transformed matrix, we
use the mask NB02 stores in the reduced pickle (`undetected_mtb`: True where the
**pre-imputation** value was missing or <= 0) and set those cells to `NaN`, so MOFA+'s
native missing-value handling -- not a Gaussian likelihood fit to an imputed value --
models them. The percentage masked is printed below.

In [ ]:
mtb_feature_cols = [c for c in mtb.columns if c != 'Sample']

und_all = _prep.get('undetected_mtb', {}).get(CRC_DS)
if und_all is None:
    raise RuntimeError('preprocessed_data_reduced_v2.pkl has no undetected_mtb mask for '
                       f'{CRC_DS} -- re-run 02_analysis.ipynb with the current version.')
und = (und_all.set_index('Sample').reindex(mtb['Sample'])
       .reindex(columns=mtb_feature_cols).fillna(False).astype(bool))

mtb_nan = mtb.copy()
mtb_nan[mtb_feature_cols] = mtb[mtb_feature_cols].mask(und.to_numpy())
mtb = mtb_nan  # downstream cells use the NaN-ified metabolite matrix

pct_col = und.mean(axis=0)
print(f'Cells converted undetected -> NaN: {int(und.to_numpy().sum())}/{und.size} '
      f'({100 * und.to_numpy().mean():.1f}%)')
print(f'Metabolites with any undetected cells: {(pct_col > 0).sum()}/{len(mtb_feature_cols)}; '
      f'median per-metabolite missingness {100 * pct_col.median():.1f}%')


## 3. Align samples and build the MOFA+ input matrices (view 0 = species, view 1 = metabolites)


In [ ]:
common_samples = sorted(set(spc['Sample']) & set(mtb['Sample']) & set(meta['Sample']))
print(f'Common samples across species/metabolite/metadata views: {len(common_samples)}')

spc_aligned  = spc.set_index('Sample').loc[common_samples]
mtb_aligned  = mtb.set_index('Sample').loc[common_samples]
meta_aligned = meta.set_index('Sample').loc[common_samples]

spc_names = list(spc_aligned.columns)
mtb_names = list(mtb_aligned.columns)

spc_mat = spc_aligned.to_numpy(dtype=float)   # n_samples x n_species; CLR-transformed, no NaN expected
mtb_mat = mtb_aligned.to_numpy(dtype=float)   # n_samples x n_metabolites; NaN = sub-detection-limit

n_nan_spc = int(np.isnan(spc_mat).sum())
if n_nan_spc:
    warnings.warn(f'{n_nan_spc} unexpected NaN cells in CLR-transformed species matrix '
                   f'(utils.clr_transform() should leave none)')

print(f'Species matrix    : {spc_mat.shape}  (view 0 -- "species")')
print(f'Metabolite matrix : {mtb_mat.shape}  (view 1 -- "metabolites"), '
      f'{np.isnan(mtb_mat).sum()} NaN cells ({100*np.isnan(mtb_mat).mean():.1f}%) '
      f'passed through to MOFA+ as missing data')

# bug #3 fix, reiterated: do NOT np.nan_to_num() mtb_mat. mofapy2 handles NaN natively via
# per-feature masks (see mofapy2.run.entry_point / process_data, which use np.nanmean etc.
# for centering); 0.0 is not a neutral value on this CLR/log-transformed scale.


## 4. Fit MOFA+ (single group, species + metabolite views) and prune to active factors

All samples form **one** MOFA+ group (see the note at the top: per-stage groups would
remove the between-stage signal the stage test looks for).

Factor count is **not** hardcoded (bug fix #4): we fit with a generous ceiling of 15
factors (ARD prior on the weights), then explicitly prune to active factors via
`removeInactiveFactors()`. That method removes only **one** factor per call by design, so
we loop it until no further factor qualifies, with a guard against ever reaching `K == 0`.

In [ ]:
# bug #8 fix: defensively confirm the pickled harmonized_meta actually matches utils's
# canonical maps (rather than silently trusting an upstream notebook run against a stale
# utils.py) -- CRC_STAGE_NUMERIC / STAGE_GROUP_MAP / STAGE_GROUP_NUMERIC are never
# redefined locally, only checked against.
_expected_crc_stage_numeric = meta_aligned['Study.Group'].map(CRC_STAGE_NUMERIC)
assert (meta_aligned['CRC.Stage.Numeric'].fillna(-1).to_numpy()
        == _expected_crc_stage_numeric.fillna(-1).to_numpy()).all(), (
    'meta CRC.Stage.Numeric does not match utils.CRC_STAGE_NUMERIC -- harmonize_metadata() '
    'output may be stale.'
)

_expected_stage_group = meta_aligned['Study.Group'].map(STAGE_GROUP_MAP)
assert (meta_aligned['Stage.Group'].fillna('NA').to_numpy()
        == _expected_stage_group.fillna('NA').to_numpy()).all(), (
    'meta Stage.Group does not match utils.STAGE_GROUP_MAP -- harmonize_metadata() output '
    'may be stale (bug fix #8: no local stage-map redefinition allowed here).'
)

_expected_stage_group_numeric = meta_aligned['Stage.Group'].map(STAGE_GROUP_NUMERIC)
assert (meta_aligned['Stage.Group.Numeric'].fillna(-1).to_numpy()
        == _expected_stage_group_numeric.fillna(-1).to_numpy()).all(), (
    'meta Stage.Group.Numeric does not match utils.STAGE_GROUP_NUMERIC -- harmonize_metadata() '
    'output may be stale.'
)

# Single MOFA+ group: per-stage groups would centre away the between-stage differences.
GROUPS = ['all']
group_sample_idx = {'all': list(common_samples)}
print(f'Single-group MOFA+: {len(common_samples)} samples')
print(meta_aligned['Study.Group'].value_counts().reindex(CRC_STAGE_ORDER).to_string())

N_FACTORS_INIT = 15   # generous ceiling; pruned below -- factor count is justified via
                       # ARD + explicit post-hoc pruning, not hardcoded (bug #4 fix)
MIN_R2_ACTIVE  = 0.02  # 2% variance explained in >=1 view -- standard MOFA+
                        # active-factor threshold; verified directly against
                        # mofapy2.core.BayesNet.removeInactiveFactors source: a factor is
                        # dropped only if its R2 is <= min_r2 in EVERY view AND EVERY group
                        # simultaneously.

MOFA_SUCCESS = False
MOFA_R2 = None

try:
    from mofapy2.run.entry_point import entry_point

    ent = entry_point()

    spc_by_group     = [spc_aligned.loc[group_sample_idx[g]].to_numpy() for g in GROUPS]
    mtb_by_group     = [mtb_aligned.loc[group_sample_idx[g]].to_numpy() for g in GROUPS]
    samples_by_group = [group_sample_idx[g] for g in GROUPS]

    # views_names fixes the view axis used everywhere below: view 0 = species, view 1 = metabolites
    ent.set_data_matrix(
        [spc_by_group, mtb_by_group],
        likelihoods=['gaussian', 'gaussian'],
        views_names=['species', 'metabolites'],
        groups_names=GROUPS,
        samples_names=samples_by_group,
        features_names=[spc_names, mtb_names],
    )
    ent.set_model_options(
        factors=N_FACTORS_INIT,
        spikeslab_weights=True,
        ard_factors=False,  # only needed for multi-group models
        ard_weights=True,   # required whenever M > 1 (multi-view); ARD prior on W is what
                             # makes removeInactiveFactors()'s pruning below meaningful
    )
    ent.set_train_options(iter=1000, convergence_mode='medium', seed=1, verbose=False)
    ent.build()
    ent.run()

    N_FACTORS_PRE_PRUNE = ent.model.dim['K']
    print(f'\nMOFA+ fit complete: {N_FACTORS_PRE_PRUNE} factors (pre-prune), '
          f'{ent.model.dim["N"]} samples total.')

    dropped_factors = []
    while ent.model.dim['K'] > 1:
        dropped_idx = np.atleast_1d(
            ent.model.removeInactiveFactors(min_r2=MIN_R2_ACTIVE, return_idx=True)
        )
        if dropped_idx.size == 0:
            break
        dropped_factors.append(int(dropped_idx[0]))

    n_factors_active = ent.model.dim['K']
    print(f'removeInactiveFactors(min_r2={MIN_R2_ACTIVE}): dropped {len(dropped_factors)} of '
          f'{N_FACTORS_PRE_PRUNE} factors -> {n_factors_active} active factors retained.')

    # Factor scores: Z dim is (N_total, K_active)
    Z_raw = np.asarray(ent.model.nodes['Z'].getExpectations()['E'])
    all_samples_ordered = [s for g in GROUPS for s in group_sample_idx[g]]
    assert Z_raw.shape == (len(all_samples_ordered), n_factors_active), (
        f'Unexpected Z shape {Z_raw.shape}, expected ({len(all_samples_ordered)}, {n_factors_active})'
    )

    # Loadings: W dim is (D_view, K_active) per view (verified against
    # mofapy2.core.nodes.W_nodes.W_Node.removeFactors -> super().removeFactors(idx, axis=1),
    # i.e. axis 1 is the factor axis, axis 0 is the feature axis)
    W_raw = ent.model.nodes['W'].getExpectations()
    W_spc = np.asarray(W_raw[0]['E'])
    W_mtb = np.asarray(W_raw[1]['E'])
    if W_spc.shape[0] != len(spc_names):
        W_spc = W_spc.T
    if W_mtb.shape[0] != len(mtb_names):
        W_mtb = W_mtb.T
    assert W_spc.shape == (len(spc_names), n_factors_active)
    assert W_mtb.shape == (len(mtb_names), n_factors_active)

    # Variance explained, post-prune (bug #2 fix -- see markdown/cell below for the indexing bug)
    MOFA_R2 = ent.model.calculate_variance_explained()  # list of length len(GROUPS); each
                                                          # element an (n_views, K_active) array

    MOFA_SUCCESS = True

except Exception as e:
    print(f'MOFA+ fitting failed ({type(e).__name__}: {e}); falling back to PCA for '
          f'demonstration purposes only -- outputs from this branch are NOT a substitute '
          f'for a real MOFA+ run.')
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    # PCA can't handle NaN -- impute per-column mean only for THIS fallback path (the
    # try branch above never sees an imputed value for these sub-detection-limit cells)
    mtb_mat_pca = np.where(np.isnan(mtb_mat), np.nanmean(mtb_mat, axis=0, keepdims=True), mtb_mat)
    combined = np.hstack([StandardScaler().fit_transform(spc_mat),
                           StandardScaler().fit_transform(mtb_mat_pca)])

    pca_full = PCA(n_components=min(N_FACTORS_INIT, combined.shape[0] - 1, combined.shape[1]),
                    random_state=1)
    Z_pca_full = pca_full.fit_transform(combined)

    n_factors_active = max(1, int((pca_full.explained_variance_ratio_ >= MIN_R2_ACTIVE).sum()))
    print(f'PCA fallback: {pca_full.n_components_} PCs fit, {n_factors_active} retained '
          f'(explained_variance_ratio_ >= {MIN_R2_ACTIVE})')

    Z_raw = Z_pca_full[:, :n_factors_active]
    all_samples_ordered = list(common_samples)
    W_spc = pca_full.components_[:n_factors_active, :spc_mat.shape[1]].T
    W_mtb = pca_full.components_[:n_factors_active, spc_mat.shape[1]:].T
    MOFA_R2 = None


## 5. Assemble factor scores


In [ ]:
factor_cols = [f'Factor{i+1}' for i in range(n_factors_active)]

Z_df = pd.DataFrame(Z_raw, columns=factor_cols, index=all_samples_ordered)
Z_df.index.name = 'Sample'
Z_df = Z_df.reset_index().merge(
    meta_aligned.reset_index()[
        ['Sample', 'Study.Group', 'CRC.Stage.Numeric', 'Stage.Group', 'Stage.Group.Numeric']
    ],
    on='Sample', how='left',
)

print(f'Factor score table: {Z_df.shape[0]} samples x {n_factors_active} factors '
      f'({"MOFA+" if MOFA_SUCCESS else "PCA fallback"})')
Z_df.head()


## 6. Sign-canonicalization (bug fix #7)

See the notebook-level markdown at the top for the full convention. Reference
column is `CRC.Stage.Numeric` (`utils.CRC_STAGE_NUMERIC`, fine-grained
corrected tiers), fully populated here since this notebook uses Yachida only.


In [ ]:
sign_log = []
for i, fc in enumerate(factor_cols):
    rho, _ = spearmanr(Z_df[fc], Z_df['CRC.Stage.Numeric'])
    flip = bool(rho < 0)
    if flip:
        Z_df[fc] = -Z_df[fc]
        W_spc[:, i] = -W_spc[:, i]
        W_mtb[:, i] = -W_mtb[:, i]
    sign_log.append({'Factor': fc, 'Rho_vs_CRC_Stage_Numeric_preflip': round(float(rho), 4),
                      'Flipped': flip})

sign_log_df = pd.DataFrame(sign_log)
print('Sign canonicalization (after this cell, positive Factor score = more advanced disease):')
print(sign_log_df.to_string(index=False))


## 7. Variance explained (bug fix #2)

`calculate_variance_explained()` returns a **list of length `len(GROUPS)`**;
each element is an `(n_views, K)` array -- indexed `[group][view]`, NOT a
dict with an `'r2_per_factor'` key. Verified directly against the installed
`mofapy2` source (`mofapy2/core/BayesNet.py`):
`r2 = [np.zeros([self.dim["M"], self.dim["K"]]) for g in range(self.dim["G"])]`.
Species is view 0, metabolites is view 1 (fixed by
`views_names=['species', 'metabolites']` above).


In [ ]:
if MOFA_SUCCESS and MOFA_R2 is not None:
    r2_spc = np.array([MOFA_R2[g][0] for g in range(len(GROUPS))])  # (n_groups, K_active)
    r2_mtb = np.array([MOFA_R2[g][1] for g in range(len(GROUPS))])  # (n_groups, K_active)

    fig, axes = plt.subplots(1, 2, figsize=(max(10, n_factors_active * 0.9), 5))
    for ax, r2_mat, view_title in zip(axes, [r2_spc, r2_mtb], ['Species', 'Metabolites']):
        df_r2 = pd.DataFrame(r2_mat, index=GROUPS, columns=factor_cols)
        sns.heatmap(df_r2, cmap='YlOrRd', ax=ax, annot=True, fmt='.2f',
                    cbar_kws={'label': 'R2'}, linewidths=0.3, annot_kws={'size': 7})
        ax.set_title(f'Variance Explained -- {view_title}', fontsize=11)
        ax.set_xlabel('Factor'); ax.set_ylabel('MOFA+ group (single group: all samples)')
        ax.tick_params(axis='x', rotation=45)
    fig.suptitle('MOFA+ Variance Explained per Factor (post-prune, sign-canonicalized)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()

    figs_dir = RESULTS_DIR / 'figures' / 'multiomics'
    figs_dir.mkdir(parents=True, exist_ok=True)   # bug #9 fix: mkdir before every savefig/to_csv
    _vfig = figs_dir / 'mofa_variance_explained_v2.png'
    fig.savefig(_vfig, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Saved: {_vfig}')
else:
    print('Variance-explained figure skipped (PCA fallback path has no per-view/group R2 decomposition).')


## 8. Factor <-> CRC stage association (bug fixes #5, #8)

Kruskal-Wallis across `Stage.Group` (Healthy / Adenoma / Early_CRC / Advanced_CRC, per
`utils.STAGE_GROUP_MAP` / `STAGE_GROUP_ORDER`) plus Spearman correlation against the
ordinal `Stage.Group.Numeric` (primary) and the fine-grained `CRC.Stage.Numeric`
(secondary), each BH-FDR corrected across all tested factors.

In [ ]:
stage_groups_present = [g for g in STAGE_GROUP_ORDER if g in Z_df['Stage.Group'].to_numpy()]

kw_rows = []
for fc in factor_cols:
    groups_vals = [Z_df.loc[Z_df['Stage.Group'] == sg, fc].dropna().to_numpy()
                   for sg in stage_groups_present]
    groups_vals = [v for v in groups_vals if len(v) > 3]
    if len(groups_vals) < 2:
        continue
    kw_stat, kw_p = kruskal(*groups_vals)

    valid = Z_df['Stage.Group.Numeric'].notna()
    rho, sp_p = spearmanr(Z_df.loc[valid, fc], Z_df.loc[valid, 'Stage.Group.Numeric'])
    valid_f = Z_df['CRC.Stage.Numeric'].notna()
    rho_f, sp_p_f = spearmanr(Z_df.loc[valid_f, fc], Z_df.loc[valid_f, 'CRC.Stage.Numeric'])

    kw_rows.append({
        'Factor': fc, 'KW_stat': kw_stat, 'KW_pval': kw_p,
        'Spearman_rho': rho, 'Spearman_pval': sp_p,
        'Spearman_rho_fine': rho_f, 'Spearman_pval_fine': sp_p_f,
        'n_samples': int(valid.sum()),
    })

kw_df = pd.DataFrame(kw_rows)
_, kw_df['KW_qval'], _, _       = multipletests(kw_df['KW_pval'],       method='fdr_bh')
_, kw_df['Spearman_qval'], _, _ = multipletests(kw_df['Spearman_pval'], method='fdr_bh')
_, kw_df['Spearman_qval_fine'], _, _ = multipletests(kw_df['Spearman_pval_fine'], method='fdr_bh')
kw_df['Significant_KW_FDR05']       = kw_df['KW_qval']       < 0.05
kw_df['Significant_Spearman_FDR05'] = kw_df['Spearman_qval'] < 0.05
kw_df = kw_df.sort_values('KW_qval')

tables_dir = RESULTS_DIR / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)   # bug #9 fix
assoc_path = tables_dir / 'mofa_factor_stage_association_v2.csv'
kw_df.to_csv(assoc_path, index=False)

print(f'Factor <-> CRC Stage.Group association (BH-FDR corrected across {len(kw_df)} factors):')
print(kw_df.to_string(index=False))
print(f'\nSaved: {assoc_path}')


## 9. Factor score boxplots by stage (top factors by KW FDR-q, for readability)


In [ ]:
# viz keeps a group's colour identical across notebooks, and gives PostSurgery the off-axis treatment rather than a severity-ramp step it has no ordinal position for.
stage_palette = viz.group_palette(stage_groups_present,
                                  order=utils.STAGE_GROUP_ORDER)
top_factors = kw_df.head(min(3, len(kw_df)))['Factor'].tolist()

if top_factors:
    fig, axes = plt.subplots(1, len(top_factors), figsize=(7 * len(top_factors), 5), squeeze=False)
    axes = axes[0]
    for ax, fc in zip(axes, top_factors):
        row = kw_df[kw_df['Factor'] == fc].iloc[0]
        data_plot = [Z_df.loc[Z_df['Stage.Group'] == sg, fc].dropna().to_numpy()
                     for sg in stage_groups_present]
        bp = ax.boxplot(data_plot, patch_artist=True,
                         medianprops=dict(color='black', linewidth=2),
                         flierprops=dict(marker='o', markersize=3, alpha=0.4))
        for patch, sg in zip(bp['boxes'], stage_groups_present):
            patch.set_facecolor(stage_palette.get(sg, '#cccccc'))
            patch.set_alpha(0.75)
        for j, (sg, vals) in enumerate(zip(stage_groups_present, data_plot)):
            ax.scatter(np.random.normal(j + 1, 0.07, size=len(vals)), vals,
                       color=stage_palette.get(sg, '#cccccc'), s=12, alpha=0.5, zorder=3)
        ax.set_xticks(range(1, len(stage_groups_present) + 1))
        ax.set_xticklabels(stage_groups_present, rotation=20, ha='right', fontsize=9)
        ax.set_ylabel(f'{fc} score (sign-canonicalized)', fontsize=10)
        ax.set_title(f"{fc}  KW q={row['KW_qval']:.4f}  |  Spearman rho={row['Spearman_rho']:+.3f} "
                     f"(q={row['Spearman_qval']:.4f})", fontsize=10)
        ax.axhline(0, color='grey', ls='--', lw=0.8, alpha=0.5)
        ax.grid(True, axis='y', alpha=0.3)

    plt.suptitle('MOFA+ Factor Scores by CRC Stage.Group '
                 f'(top {len(top_factors)} of {len(kw_df)} tested factors by KW FDR-q)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()

    figs_dir = RESULTS_DIR / 'figures' / 'multiomics'
    figs_dir.mkdir(parents=True, exist_ok=True)   # bug #9 fix
    out_fig = figs_dir / 'mofa_factor_scores_by_stage_v2.png'
    fig.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Saved: {out_fig}')
else:
    print('No factor had >=2 stage groups with n>3 samples -- boxplot skipped.')


## 10. Factor biplot (top-2 CRC-associated factors, coloured by stage)


In [ ]:
if len(top_factors) >= 2:
    f1, f2 = top_factors[0], top_factors[1]
    fig, ax = plt.subplots(figsize=(8, 7))
    for sg in stage_groups_present:
        sub = Z_df[Z_df['Stage.Group'] == sg]
        ax.scatter(sub[f1], sub[f2], c=stage_palette.get(sg, '#cccccc'),
                  label=f'{sg} (n={len(sub)})', s=28, alpha=0.65, edgecolors='none')
    ax.axhline(0, color='grey', lw=0.7, ls='--')
    ax.axvline(0, color='grey', lw=0.7, ls='--')
    ax.set_xlabel(f'{f1} score'); ax.set_ylabel(f'{f2} score')
    ax.set_title(f'MOFA+ Factor Biplot ({f1} vs {f2}), coloured by Stage.Group')
    ax.legend(title='Stage.Group', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()

    figs_dir = RESULTS_DIR / 'figures' / 'multiomics'
    figs_dir.mkdir(parents=True, exist_ok=True)   # bug #9 fix
    _bfig = figs_dir / 'mofa_factor_biplot_v2.png'
    fig.savefig(_bfig, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Saved: {_bfig}')
else:
    print('Fewer than 2 tested factors -- biplot skipped.')


## 11. Full factor loadings -- ALL active factors x ALL features (bug fix #6)

This is the hard NB08 contract requirement: NB08 hardcodes specific `FactorN`
values and needs them to actually exist in this file regardless of MOFA+'s
arbitrary internal factor ordering. No top-3/top-10 filtering here -- every
active factor, every species, every metabolite.


In [ ]:
loading_rows = []
for i, fc in enumerate(factor_cols):
    for feat, w in zip(spc_names, W_spc[:, i]):
        loading_rows.append({'Factor': fc, 'View': 'species', 'Feature': feat, 'Weight': float(w)})
    for feat, w in zip(mtb_names, W_mtb[:, i]):
        loading_rows.append({'Factor': fc, 'View': 'metabolites', 'Feature': feat, 'Weight': float(w)})

loadings_df = pd.DataFrame(loading_rows)

tables_dir = RESULTS_DIR / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)   # bug #9 fix
loadings_path = tables_dir / 'mofa_factor_loadings_v2.csv'
loadings_df.to_csv(loadings_path, index=False)

print(f'Loadings saved: {len(loadings_df)} rows = {n_factors_active} factors x '
      f'({len(spc_names)} species + {len(mtb_names)} metabolites)')
print(f'Saved: {loadings_path}')

# Polyamine loading check across ALL factors (not just the KW-top-3)
pa_name_set = set(pa_cols.values())
pa_loadings = loadings_df[(loadings_df['View'] == 'metabolites')
                           & (loadings_df['Feature'].isin(pa_name_set))].copy()
pa_loadings = pa_loadings.reindex(pa_loadings['Weight'].abs().sort_values(ascending=False).index)
print(f'\nPolyamine loadings across all {n_factors_active} factors ({len(pa_loadings)} rows), top 15 by |weight|:')
print(pa_loadings.head(15).to_string(index=False))


## 12. Loading heatmap (top features per view, across all active factors -- visualization only; the table above has everything)


In [ ]:
_TOP_FEAT = 20
_top_spc_feat = pd.Series(np.abs(W_spc).max(axis=1), index=spc_names).nlargest(_TOP_FEAT).index.tolist()
_top_mtb_feat = pd.Series(np.abs(W_mtb).max(axis=1), index=mtb_names).nlargest(_TOP_FEAT).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, W, feat_names, top_feats, title in [
    (axes[0], W_spc, spc_names, _top_spc_feat, 'Species Loadings'),
    (axes[1], W_mtb, mtb_names, _top_mtb_feat, 'Metabolite Loadings'),
]:
    idx = [feat_names.index(f) for f in top_feats]
    df_w = pd.DataFrame(W[idx, :], index=[feat_names[i] for i in idx], columns=factor_cols)
    sns.heatmap(df_w, cmap='RdBu_r', center=0, ax=ax, linewidths=0.2,
                cbar_kws={'label': 'MOFA+ weight (sign-canonicalized)'}, yticklabels=True)
    ax.set_title(title, fontsize=11)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ycolors = ['#e74c3c' if f in pa_name_set else 'black' for f in df_w.index]
    for lbl, col in zip(ax.get_yticklabels(), ycolors):
        lbl.set_color(col)
    ax.tick_params(axis='y', labelsize=7)

fig.suptitle(f'MOFA+ Top {_TOP_FEAT} Feature Loadings per View, all {n_factors_active} active factors\n'
             '(red labels = polyamine metabolites; this figure subsets for readability -- '
             'mofa_factor_loadings_v2.csv has every feature)', fontsize=11, fontweight='bold')
plt.tight_layout()

figs_dir = RESULTS_DIR / 'figures' / 'multiomics'
figs_dir.mkdir(parents=True, exist_ok=True)   # bug #9 fix
_lfig = figs_dir / 'mofa_loading_heatmap_v2.png'
fig.savefig(_lfig, dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Saved: {_lfig}')


## 13. Save factor scores

`Sample` + every `FactorN` (sign-canonicalized) + `Study.Group` +
`CRC.Stage.Numeric` + `Stage.Group` (+ `Stage.Group.Numeric` as a bonus
column beyond the minimum contract).


In [ ]:
tables_dir = RESULTS_DIR / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)   # bug #9 fix
scores_path = tables_dir / 'mofa_factor_scores_v2.csv'
Z_df.to_csv(scores_path, index=False)
print(f'Saved factor scores: {scores_path}')
print(f'Columns: {list(Z_df.columns)}')


## Summary


In [ ]:
print('=== NB06 MOFA+ Analysis Summary ===')
print(f'\nModel: {"MOFA+ (mofapy2)" if MOFA_SUCCESS else "PCA fallback"}')
print(f'Active factors: {n_factors_active}, Samples: {len(common_samples)}')
print(f'\nFactors significant vs Stage.Group after BH-FDR (q<0.05):')
_sig = kw_df[kw_df['Significant_KW_FDR05']]
print(_sig[['Factor', 'KW_qval', 'Spearman_rho', 'Spearman_qval']].to_string(index=False)
      if not _sig.empty else '  (none)')
print('\nOutputs:')
print(f'  - tables/mofa_factor_stage_association_v2.csv  ({len(kw_df)} factors)')
print(f'  - tables/mofa_factor_loadings_v2.csv            ({len(loadings_df)} rows, '
      f'{n_factors_active} factors x {len(spc_names)+len(mtb_names)} features)')
print(f'  - tables/mofa_factor_scores_v2.csv              ({len(Z_df)} samples)')
print('  - figures/multiomics/mofa_variance_explained_v2.png')
print('  - figures/multiomics/mofa_factor_scores_by_stage_v2.png')
print('  - figures/multiomics/mofa_factor_biplot_v2.png')
print('  - figures/multiomics/mofa_loading_heatmap_v2.png')
